# Week 1 — Data Pipeline & Baseline Statistics

**Project:** Vectorial vs. Alignment-Based Models of Locust Collective Motion  
**Data:** Weinburd et al. (2024) — ~20,000 locust trajectories from Australian plague locust hopper bands  
**Goal:** Load the field data and compute four baseline metrics (order parameter, turning angles, neighbor density maps, nearest-neighbor distances) that we will later compare against simulated Vicsek and Pull model outputs.

**Data source:** [Dryad dataset](https://doi.org/10.5061/dryad.n02v6wwzz) | [Analysis code](https://github.com/weinburd/locust_trajectory_data)

In [ ]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (12, 8),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# Feature column indices (0-based, converted from MATLAB 1-based)
IDX_X = 0
IDX_Y = 1
IDX_FLAG = 2
IDX_SPEED = 3
IDX_THETA = 4
IDX_LOCAL_SPEED = 5
IDX_LOCAL_STD = 6
IDX_LOCAL_MINMAX = 7
IDX_STATE = 8

# Constants
FPS = 25
DT = 1.0 / FPS  # 0.04 seconds per frame

print("Imports and constants ready.")

## 1. Load & Convert Data

The Weinburd dataset stores trajectories as MATLAB struct arrays with shape `(N_locusts, N_frames)`, where each element contains a `features` vector of length 9 and a `neighbors` list. We convert these to dense 3D NumPy arrays of shape `(N_locusts, 9, N_frames)`, filling missing entries with NaN.

Two `.mat` files split the four hopper bands:
- `data_recording.mat` → recordings 0-1 (bands 1 & 3)
- `data_recording2.mat` → recordings 2-3 (bands 2 & 6)

In [ ]:
def struct2data(data_struct):
    """Convert MATLAB struct array to dense 3D NumPy array.
    
    Mirrors the logic of Weinburd's struct2data.m.
    
    Args:
        data_struct: ndarray of shape (N_locusts, N_frames) with dtype containing 'features' field
    
    Returns:
        data_final: ndarray of shape (N_locusts, 9, N_frames), NaN where no data
    """
    N_locusts, N_frames = data_struct.shape
    data_final = np.full((N_locusts, 9, N_frames), np.nan, dtype=np.float64)
    
    for i in range(N_locusts):
        for j in range(N_frames):
            feat = data_struct[i, j]['features']
            if feat.size > 0:
                data_final[i, :, j] = feat.flatten()
    
    return data_final


def load_all_clips(mat_path, rec_key='recording'):
    """Load all clips from a .mat file.
    
    Returns:
        clips: list of dicts, each with keys:
            'name': clip identifier string
            'data': 3D array (N_locusts, 9, N_frames)
            'scale': pixels/cm
            'fieldDims': [xmin, xmax, ymin, ymax] in pixels
            'area_m2': arena area in m^2
    """
    mat = scipy.io.loadmat(mat_path, squeeze_me=False)
    rec = mat[rec_key]
    
    clips = []
    for r in range(rec.shape[1]):
        rec_data = rec[0, r]
        data_field = rec_data['data']
        
        # Get scale — some recordings use corners (projective transform) instead
        scale_arr = rec_data['scale']
        field_dims = rec_data['fieldDims'].flatten()
        
        if scale_arr.size > 0:
            scale = float(scale_arr.flatten()[0])
            # Arena dimensions in cm
            width_cm = field_dims[1] / scale
            height_cm = field_dims[3] / scale
            area_m2 = (width_cm * height_cm) * 1e-4  # cm^2 -> m^2
        else:
            # Recordings with corners need projective transform
            # Use approximate scale from the data itself
            scale = 18.0  # approximate
            width_cm = field_dims[1] / scale
            height_cm = field_dims[3] / scale
            area_m2 = (width_cm * height_cm) * 1e-4
        
        n_clips = data_field.shape[0]
        for c in range(n_clips):
            clip_name = str(data_field[c, 0].flatten()[0])
            clip_struct = data_field[c, 1]
            
            print(f"  Converting clip '{clip_name}' ({clip_struct.shape[0]} locusts x {clip_struct.shape[1]} frames)...")
            data_3d = struct2data(clip_struct)
            
            clips.append({
                'name': clip_name,
                'data': data_3d,
                'scale': scale,
                'field_dims_pix': field_dims,
                'width_cm': width_cm,
                'height_cm': height_cm,
                'area_m2': area_m2,
                'recording_idx': r,
            })
    
    return clips

print("Loading functions defined.")

In [ ]:
DATA_DIR = '../data/'

print("Loading data_recording.mat (bands 1 & 3)...")
clips_1 = load_all_clips(DATA_DIR + 'data_recording.mat', rec_key='recording')

print(f"\nLoading data_recording2.mat (bands 2 & 6)...")
clips_2 = load_all_clips(DATA_DIR + 'data_recording2.mat', rec_key='recording2')

# Combine — recordings 0-1 come from file 1, recordings 2-3 from file 2
# But file 2 has duplicates of recordings 0-1 (same structure). 
# Per Weinburd's fig2.m: file_idx 1-2 -> recording, file_idx 3-4 -> recording2
# So we take recordings 0-1 from file 1, recordings 2-3 from file 2
all_clips = [c for c in clips_1 if c['recording_idx'] <= 1] + \
            [c for c in clips_2 if c['recording_idx'] >= 2]

print(f"\n=== Total: {len(all_clips)} clips loaded ===")

In [ ]:
# Summary statistics for each clip
band_names = {0: 'Band 1 (vid133)', 1: 'Band 3 (vid098)', 2: 'Band 2 (vid096)', 3: 'Band 6 (vid146)'}

print(f"{'Clip':<25} {'Band':<20} {'Locusts':>8} {'Frames':>8} {'Valid Obs':>12} {'Density':>12} {'Arena (cm)':>15}")
print("-" * 105)

total_obs = 0
total_locusts = 0
for clip in all_clips:
    d = clip['data']
    n_locusts, _, n_frames = d.shape
    valid = np.sum(d[:, IDX_FLAG, :] == 1)
    total_obs += valid
    total_locusts += n_locusts
    
    # Mean density: average number of valid locusts per frame / area
    valid_per_frame = np.sum(d[:, IDX_FLAG, :] == 1, axis=0)
    mean_density = np.mean(valid_per_frame) / clip['area_m2']
    
    band = band_names.get(clip['recording_idx'], f"Rec {clip['recording_idx']}")
    arena = f"{clip['width_cm']:.1f} x {clip['height_cm']:.1f}"
    
    print(f"{clip['name']:<25} {band:<20} {n_locusts:>8} {n_frames:>8} {valid:>12,} {mean_density:>10.1f}/m² {arena:>15}")

print(f"\n{'TOTAL':<25} {'':<20} {total_locusts:>8} {'':<8} {total_obs:>12,}")
print(f"\nTotal unique trajectories: {total_locusts:,}")
print(f"Total valid position observations: {total_obs:,}")

## 2. Trajectory Extraction & Validation

Extract per-locust trajectories and verify the data makes physical sense: positions within the arena, speeds in a reasonable range, headings in $[-\pi, \pi]$.

In [ ]:
# Pick a representative clip to validate
clip = all_clips[0]
d = clip['data']  # (N_locusts, 9, N_frames)

# Find a locust with a long track
valid_counts = np.sum(d[:, IDX_FLAG, :] == 1, axis=1)
best_locust = np.argmax(valid_counts)
print(f"Clip: {clip['name']}")
print(f"Longest track: locust {best_locust} with {valid_counts[best_locust]} valid frames ({valid_counts[best_locust]*DT:.1f}s)")
print(f"Track length distribution: mean={np.mean(valid_counts):.1f}, median={np.median(valid_counts):.0f}, max={np.max(valid_counts)}")

# Extract this locust's trajectory
traj = d[best_locust]  # (9, N_frames)
valid_mask = traj[IDX_FLAG] == 1
t_valid = np.where(valid_mask)[0]

x = traj[IDX_X, valid_mask]
y = traj[IDX_Y, valid_mask]
speed = traj[IDX_SPEED, valid_mask]
theta = traj[IDX_THETA, valid_mask]
state = traj[IDX_STATE, valid_mask]

print(f"\nPosition range: x=[{x.min():.1f}, {x.max():.1f}] cm, y=[{y.min():.1f}, {y.max():.1f}] cm")
print(f"Speed range: [{np.nanmin(speed):.2f}, {np.nanmax(speed):.2f}] cm/s, mean={np.nanmean(speed):.2f} cm/s")
print(f"Heading range: [{np.nanmin(theta):.3f}, {np.nanmax(theta):.3f}] rad")
print(f"Motion states: stationary={np.sum(state==0)}, walking={np.sum(state==1)}, hopping={np.sum(state==2)}, NaN={np.sum(np.isnan(state))}")

# Plot example trajectory
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Trajectory colored by time
sc = axes[0].scatter(x, y, c=t_valid * DT, cmap='viridis', s=5, alpha=0.7)
axes[0].set_xlabel('X (cm)')
axes[0].set_ylabel('Y (cm)')
axes[0].set_title(f'Trajectory of locust {best_locust}')
axes[0].set_aspect('equal')
plt.colorbar(sc, ax=axes[0], label='Time (s)')

# Speed over time
axes[1].plot(t_valid * DT, speed, lw=0.8)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Speed (cm/s)')
axes[1].set_title('Instantaneous speed')

# Heading over time
axes[2].plot(t_valid * DT, theta, lw=0.8)
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Heading (rad)')
axes[2].set_title('Heading angle')

plt.tight_layout()
plt.show()

## 3. Metric 1 — Order Parameter (Polarization)

The polarization $\Phi(t) = \left| \frac{1}{N(t)} \sum_{i=1}^{N(t)} e^{i\theta_i(t)} \right|$ measures global alignment. $\Phi = 0$ means random headings, $\Phi = 1$ means perfect alignment. This is the key metric that Buhl et al. (2006) used to claim a density-driven phase transition in locusts, and what Sayin et al. (2025) argue is better explained by the pull mechanism.

In [ ]:
def compute_polarization(data):
    """Compute per-frame polarization for a clip.
    
    Matches Weinburd's collectiveData: uses unit velocity vectors,
    excludes stationary locusts (speed=0 -> NaN heading).
    
    Args:
        data: 3D array (N_locusts, 9, N_frames)
    Returns:
        polarization: 1D array (N_frames,)
        avg_direction: 1D array (N_frames,) — mean heading angle
        density_per_frame: 1D array (N_frames,) — number of valid locusts
    """
    N_locusts, _, N_frames = data.shape
    polarization = np.full(N_frames, np.nan)
    avg_direction = np.full(N_frames, np.nan)
    density_per_frame = np.zeros(N_frames)
    
    for t in range(N_frames):
        theta = data[:, IDX_THETA, t]
        speed = data[:, IDX_SPEED, t]
        flag = data[:, IDX_FLAG, t]
        
        # Valid: flag==1, non-NaN theta, non-zero speed (so heading is defined)
        valid = (flag == 1) & ~np.isnan(theta) & (speed > 0)
        n_valid = np.sum(valid)
        density_per_frame[t] = np.sum(flag == 1)
        
        if n_valid > 0:
            angles = theta[valid]
            mean_vec = np.mean(np.exp(1j * angles))
            polarization[t] = np.abs(mean_vec)
            avg_direction[t] = np.angle(mean_vec)
    
    return polarization, avg_direction, density_per_frame


# Compute for all clips
all_polarization = []
all_density = []

for clip in all_clips:
    pol, avg_dir, dens = compute_polarization(clip['data'])
    all_polarization.append(pol)
    all_density.append(dens / clip['area_m2'])  # convert to density per m^2

# Concatenate across all clips for aggregate statistics
pol_all = np.concatenate(all_polarization)
dens_all = np.concatenate(all_density)

print(f"Polarization: mean={np.nanmean(pol_all):.3f}, std={np.nanstd(pol_all):.3f}")
print(f"Density: mean={np.nanmean(dens_all):.1f} locusts/m², std={np.nanstd(dens_all):.1f}")

In [ ]:
# Plot polarization results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Time series for a few clips
for i, clip in enumerate(all_clips[:4]):
    t = np.arange(len(all_polarization[i])) * DT
    axes[0].plot(t, all_polarization[i], alpha=0.6, lw=0.8, 
                 label=f"{clip['name'][:15]}")
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Polarization $\Phi$')
axes[0].set_title('Polarization time series (first 4 clips)')
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=8)

# Histogram of polarization
axes[1].hist(pol_all[~np.isnan(pol_all)], bins=50, density=True, 
             color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(np.nanmean(pol_all), color='red', ls='--', label=f'mean={np.nanmean(pol_all):.3f}')
axes[1].set_xlabel('Polarization $\Phi$')
axes[1].set_ylabel('Density')
axes[1].set_title('Polarization distribution (all clips)')
axes[1].legend()

# Density vs polarization (scatter, as in Weinburd fig 2)
valid = ~np.isnan(pol_all) & ~np.isnan(dens_all)
axes[2].scatter(dens_all[valid][::5], pol_all[valid][::5], s=3, alpha=0.3, c='steelblue')
axes[2].set_xlabel('Density (locusts/m²)')
axes[2].set_ylabel('Polarization $\Phi$')
axes[2].set_title('Density vs Polarization')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('figures/metric1_polarization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/metric1_polarization.png")

## 4. Metric 2 — Speed & Turning-Angle Distributions

Speed distributions reveal the behavioral modes (stationary / walking / hopping). Turning-angle distributions are critical for distinguishing the Vicsek model (which predicts symmetric, noise-driven turns) from the Pull model (which predicts turns biased toward neighbors' positions).

In [ ]:
def compute_speed_and_turning(data):
    """Compute speed distributions and turning angles from a clip.
    
    Turning angle = change in heading between consecutive valid frames
    for the same locust, wrapped to [-pi, pi].
    
    Returns:
        speeds_by_state: dict {0: array, 1: array, 2: array} of speeds per motion state
        all_speeds: flat array of all valid speeds
        turning_angles: flat array of all turning angles
        turning_by_state: dict of turning angles per motion state
    """
    N_locusts, _, N_frames = data.shape
    
    speeds_by_state = {0: [], 1: [], 2: []}
    all_speeds = []
    turning_angles = []
    turning_by_state = {0: [], 1: [], 2: []}
    
    for i in range(N_locusts):
        flag = data[i, IDX_FLAG, :]
        speed = data[i, IDX_SPEED, :]
        theta = data[i, IDX_THETA, :]
        state = data[i, IDX_STATE, :]
        
        valid = (flag == 1)
        valid_idx = np.where(valid)[0]
        
        if len(valid_idx) == 0:
            continue
        
        # Collect speeds by state
        for t in valid_idx:
            s = speed[t]
            st = state[t]
            if not np.isnan(s):
                all_speeds.append(s)
                if st in (0, 1, 2):
                    speeds_by_state[int(st)].append(s)
        
        # Turning angles: consecutive valid frames only
        for k in range(len(valid_idx) - 1):
            t0, t1 = valid_idx[k], valid_idx[k + 1]
            # Only use consecutive frames (no gaps)
            if t1 - t0 != 1:
                continue
            th0, th1 = theta[t0], theta[t1]
            if np.isnan(th0) or np.isnan(th1):
                continue
            
            dtheta = th1 - th0
            # Wrap to [-pi, pi]
            dtheta = (dtheta + np.pi) % (2 * np.pi) - np.pi
            turning_angles.append(dtheta)
            
            st = state[t1]
            if st in (0, 1, 2):
                turning_by_state[int(st)].append(dtheta)
    
    return (
        {k: np.array(v) for k, v in speeds_by_state.items()},
        np.array(all_speeds),
        np.array(turning_angles),
        {k: np.array(v) for k, v in turning_by_state.items()},
    )


# Aggregate across all clips
all_speeds_agg = []
speeds_by_state_agg = {0: [], 1: [], 2: []}
all_turning_agg = []
turning_by_state_agg = {0: [], 1: [], 2: []}

for clip in all_clips:
    sbs, aspd, ta, tbs = compute_speed_and_turning(clip['data'])
    all_speeds_agg.append(aspd)
    all_turning_agg.append(ta)
    for k in (0, 1, 2):
        speeds_by_state_agg[k].append(sbs[k])
        turning_by_state_agg[k].append(tbs[k])

all_speeds_flat = np.concatenate(all_speeds_agg)
all_turning_flat = np.concatenate(all_turning_agg)
for k in (0, 1, 2):
    speeds_by_state_agg[k] = np.concatenate(speeds_by_state_agg[k])
    turning_by_state_agg[k] = np.concatenate(turning_by_state_agg[k])

state_names = {0: 'Stationary', 1: 'Walking', 2: 'Hopping'}
state_colors = {0: '#2ecc71', 1: '#3498db', 2: '#e74c3c'}

print(f"Total speed observations: {len(all_speeds_flat):,}")
print(f"Total turning angle observations: {len(all_turning_flat):,}")
for k in (0, 1, 2):
    print(f"  {state_names[k]}: {len(speeds_by_state_agg[k]):,} speeds, {len(turning_by_state_agg[k]):,} turns, "
          f"mean speed={np.mean(speeds_by_state_agg[k]):.2f} cm/s")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Speed distribution — all
axes[0, 0].hist(all_speeds_flat, bins=100, range=(0, 50), density=True,
                color='gray', edgecolor='white', alpha=0.7)
axes[0, 0].set_xlabel('Speed (cm/s)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Speed distribution (all states)')
axes[0, 0].axvline(np.median(all_speeds_flat), color='red', ls='--', 
                    label=f'median={np.median(all_speeds_flat):.1f}')
axes[0, 0].legend()

# Speed distribution — by state
for k in (0, 1, 2):
    axes[0, 1].hist(speeds_by_state_agg[k], bins=100, range=(0, 50), density=True,
                    color=state_colors[k], alpha=0.5, label=state_names[k])
axes[0, 1].set_xlabel('Speed (cm/s)')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('Speed distribution by motion state')
axes[0, 1].legend()

# Turning angle distribution — all
axes[1, 0].hist(all_turning_flat, bins=100, range=(-np.pi, np.pi), density=True,
                color='gray', edgecolor='white', alpha=0.7)
axes[1, 0].set_xlabel('Turning angle $\Delta\\theta$ (rad)')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Turning angle distribution (all states)')
axes[1, 0].axvline(0, color='red', ls='--', alpha=0.5)

# Turning angle — by state (walking and hopping only, stationary has undefined heading)
for k in (1, 2):
    if len(turning_by_state_agg[k]) > 0:
        axes[1, 1].hist(turning_by_state_agg[k], bins=100, range=(-np.pi, np.pi), density=True,
                        color=state_colors[k], alpha=0.5, label=state_names[k])
axes[1, 1].set_xlabel('Turning angle $\Delta\\theta$ (rad)')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Turning angle by motion state')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('figures/metric2_speed_turning.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/metric2_speed_turning.png")

## 5. Metric 3 — Neighbor Density Maps

For each focal locust, we rotate all neighbors within a radius into the focal locust's body-centered frame (heading = "up"). The resulting 2D density map shows where neighbors tend to be relative to a moving locust. Weinburd et al.'s main finding was that walking/hopping locusts show an anisotropic low-density zone ahead — this is key evidence for directional interactions.

In [ ]:
def compute_neighbor_density_map(data, radius=10.0, nbins=50, state_filter=None):
    """Compute body-centered neighbor density map.
    
    For each focal locust at each frame, finds neighbors within `radius` cm,
    rotates their relative positions so the focal heading points up (+y),
    and accumulates into a 2D histogram.
    
    Args:
        data: 3D array (N_locusts, 9, N_frames)
        radius: max neighbor distance in cm
        nbins: histogram resolution
        state_filter: if set (0, 1, or 2), only use focal locusts in that state
    
    Returns:
        hist: 2D array (nbins, nbins) — neighbor count density
        edges_x, edges_y: bin edges
    """
    N_locusts, _, N_frames = data.shape
    
    edges = np.linspace(-radius, radius, nbins + 1)
    hist = np.zeros((nbins, nbins))
    
    for t in range(N_frames):
        x = data[:, IDX_X, t]
        y = data[:, IDX_Y, t]
        theta = data[:, IDX_THETA, t]
        flag = data[:, IDX_FLAG, t]
        speed = data[:, IDX_SPEED, t]
        state = data[:, IDX_STATE, t]
        
        valid = (flag == 1) & ~np.isnan(x) & ~np.isnan(y)
        valid_idx = np.where(valid)[0]
        
        if len(valid_idx) < 2:
            continue
        
        pos = np.column_stack((x[valid_idx], y[valid_idx]))
        
        for ii, focal in enumerate(valid_idx):
            # Filter focal by state and valid heading
            if np.isnan(theta[focal]) or speed[focal] <= 0:
                continue
            if state_filter is not None and state[focal] != state_filter:
                continue
            
            focal_pos = pos[ii]
            focal_theta = theta[focal]
            
            # Relative positions
            rel = pos - focal_pos  # (N_valid, 2)
            
            # Distances
            dists = np.sqrt(rel[:, 0]**2 + rel[:, 1]**2)
            
            # Exclude self and beyond radius
            in_range = (dists > 0.1) & (dists < radius)
            
            if not np.any(in_range):
                continue
            
            rel_in = rel[in_range]
            
            # Rotate so focal heading points up (+y direction)
            # Rotation by -(theta - pi/2) = pi/2 - theta
            rot_angle = np.pi / 2 - focal_theta
            cos_r, sin_r = np.cos(rot_angle), np.sin(rot_angle)
            rotated_x = rel_in[:, 0] * cos_r - rel_in[:, 1] * sin_r
            rotated_y = rel_in[:, 0] * sin_r + rel_in[:, 1] * cos_r
            
            h, _, _ = np.histogram2d(rotated_x, rotated_y, bins=edges)
            hist += h
    
    return hist, edges, edges

print("Neighbor density map function defined.")

In [ ]:
# Compute neighbor density maps for each motion state
# Use a subset of clips for speed (this is computationally expensive)
# We'll use the first 6 clips (Band 1) as a representative sample

sample_clips = all_clips[:6]
RADIUS = 10.0  # cm

print("Computing neighbor density maps (this may take a few minutes)...")

density_maps = {}
for state_val, state_name in state_names.items():
    print(f"  {state_name}...")
    combined_hist = None
    for clip in sample_clips:
        h, ex, ey = compute_neighbor_density_map(clip['data'], radius=RADIUS, nbins=50, state_filter=state_val)
        if combined_hist is None:
            combined_hist = h
        else:
            combined_hist += h
    density_maps[state_val] = (combined_hist, ex, ey)

# Also compute for all moving locusts (walking + hopping)
print("  All moving...")
combined_moving = density_maps[1][0] + density_maps[2][0]
density_maps['moving'] = (combined_moving, ex, ey)

print("Done.")

In [ ]:
# Plot neighbor density maps
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

titles = ['Stationary', 'Walking', 'Hopping', 'All Moving']
keys = [0, 1, 2, 'moving']

for ax, key, title in zip(axes, keys, titles):
    hist, ex, ey = density_maps[key]
    # Normalize to density
    hist_norm = hist / (hist.sum() + 1e-10)
    
    im = ax.imshow(hist_norm.T, origin='lower', extent=[ex[0], ex[-1], ey[0], ey[-1]],
                   cmap='hot', aspect='equal')
    ax.set_xlabel('Left ← → Right (cm)')
    ax.set_ylabel('Behind ← → Ahead (cm)')
    ax.set_title(f'{title}')
    ax.axhline(0, color='white', ls='--', alpha=0.3)
    ax.axvline(0, color='white', ls='--', alpha=0.3)
    
    # Mark the focal locust position
    ax.plot(0, 0, 'w^', markersize=10)
    
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Relative density')

plt.suptitle('Body-centered neighbor density maps (focal heading = up)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/metric3_neighbor_density.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/metric3_neighbor_density.png")

## 6. Metric 4 — Nearest-Neighbor Distances

The nearest-neighbor distance (NND) distribution characterizes spacing within the swarm. This will help us calibrate the interaction radius parameter in both models.

In [ ]:
def compute_nnd(data):
    """Compute nearest-neighbor distances for each locust at each frame.
    
    Returns:
        nnd_all: flat array of all NND values (cm)
        nnd_per_frame: list of per-frame mean NND
    """
    N_locusts, _, N_frames = data.shape
    nnd_all = []
    nnd_per_frame = []
    
    for t in range(N_frames):
        x = data[:, IDX_X, t]
        y = data[:, IDX_Y, t]
        flag = data[:, IDX_FLAG, t]
        
        valid = (flag == 1) & ~np.isnan(x) & ~np.isnan(y)
        valid_idx = np.where(valid)[0]
        
        if len(valid_idx) < 2:
            nnd_per_frame.append(np.nan)
            continue
        
        pos = np.column_stack((x[valid_idx], y[valid_idx]))
        
        # Pairwise distances
        dists = cdist(pos, pos)
        np.fill_diagonal(dists, np.inf)  # exclude self
        
        min_dists = np.min(dists, axis=1)
        nnd_all.extend(min_dists)
        nnd_per_frame.append(np.mean(min_dists))
    
    return np.array(nnd_all), np.array(nnd_per_frame)


# Compute for all clips
all_nnd = []
all_nnd_timeseries = []

for clip in all_clips:
    nnd_vals, nnd_ts = compute_nnd(clip['data'])
    all_nnd.append(nnd_vals)
    all_nnd_timeseries.append(nnd_ts)

nnd_flat = np.concatenate(all_nnd)
print(f"Total NND observations: {len(nnd_flat):,}")
print(f"NND: mean={np.mean(nnd_flat):.2f} cm, median={np.median(nnd_flat):.2f} cm, "
      f"std={np.std(nnd_flat):.2f} cm")
print(f"NND percentiles: 5th={np.percentile(nnd_flat, 5):.2f}, 25th={np.percentile(nnd_flat, 25):.2f}, "
      f"75th={np.percentile(nnd_flat, 75):.2f}, 95th={np.percentile(nnd_flat, 95):.2f} cm")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NND histogram
axes[0].hist(nnd_flat, bins=100, range=(0, 15), density=True,
             color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(np.mean(nnd_flat), color='red', ls='--', 
                label=f'mean={np.mean(nnd_flat):.2f} cm')
axes[0].axvline(np.median(nnd_flat), color='orange', ls='--', 
                label=f'median={np.median(nnd_flat):.2f} cm')
axes[0].set_xlabel('Nearest-neighbor distance (cm)')
axes[0].set_ylabel('Density')
axes[0].set_title('NND distribution (all clips)')
axes[0].legend()

# NND time series for a few clips
for i, clip in enumerate(all_clips[:4]):
    t = np.arange(len(all_nnd_timeseries[i])) * DT
    axes[1].plot(t, all_nnd_timeseries[i], alpha=0.6, lw=0.8,
                 label=f"{clip['name'][:15]}")
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Mean NND (cm)')
axes[1].set_title('Mean NND over time (first 4 clips)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('figures/metric4_nnd.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/metric4_nnd.png")

## 7. Summary Statistics Export

Save all computed statistics to a markdown file for reference.

In [ ]:
summary = f"""# Week 1 — Baseline Data Statistics
## Weinburd et al. (2024) Locust Trajectory Dataset

**Generated:** from `week1_data_pipeline.ipynb`

---

## Dataset Overview

| Property | Value |
|----------|-------|
| Species | Australian plague locust (*Chortoicetes terminifera*) |
| Total clips | {len(all_clips)} |
| Hopper bands | 4 (vid133, vid098, vid096, vid146) |
| Frame rate | {FPS} fps (dt = {DT} s) |
| Frames per clip | 1,500 (60 seconds) |
| Total trajectories | {total_locusts:,} |
| Total valid observations | {total_obs:,} |

## Per-Clip Summary

| Clip | Band | Locusts | Valid Obs | Density (locusts/m²) | Arena (cm) |
|------|------|---------|-----------|---------------------|------------|
"""

for clip in all_clips:
    d = clip['data']
    n_l, _, n_f = d.shape
    v = int(np.sum(d[:, IDX_FLAG, :] == 1))
    vpf = np.sum(d[:, IDX_FLAG, :] == 1, axis=0)
    md = np.mean(vpf) / clip['area_m2']
    bn = band_names.get(clip['recording_idx'], f"Rec {clip['recording_idx']}")
    arena = f"{clip['width_cm']:.1f} x {clip['height_cm']:.1f}"
    summary += f"| {clip['name']} | {bn} | {n_l} | {v:,} | {md:.1f} | {arena} |\n"

summary += f"""
## Metric 1 — Polarization (Order Parameter)

| Statistic | Value |
|-----------|-------|
| Mean | {np.nanmean(pol_all):.4f} |
| Std | {np.nanstd(pol_all):.4f} |
| Median | {np.nanmedian(pol_all):.4f} |
| Min | {np.nanmin(pol_all):.4f} |
| Max | {np.nanmax(pol_all):.4f} |

## Metric 2 — Speed & Turning Angles

### Speed (cm/s)

| State | Count | Mean | Median | Std |
|-------|-------|------|--------|-----|
| All | {len(all_speeds_flat):,} | {np.mean(all_speeds_flat):.2f} | {np.median(all_speeds_flat):.2f} | {np.std(all_speeds_flat):.2f} |
"""

for k in (0, 1, 2):
    s = speeds_by_state_agg[k]
    summary += f"| {state_names[k]} | {len(s):,} | {np.mean(s):.2f} | {np.median(s):.2f} | {np.std(s):.2f} |\n"

summary += f"""
### Turning Angles (rad)

| State | Count | Mean | Std | Circular Std |
|-------|-------|------|-----|-------------|
| All | {len(all_turning_flat):,} | {np.mean(all_turning_flat):.4f} | {np.std(all_turning_flat):.4f} | — |
"""

for k in (1, 2):
    ta = turning_by_state_agg[k]
    summary += f"| {state_names[k]} | {len(ta):,} | {np.mean(ta):.4f} | {np.std(ta):.4f} | — |\n"

summary += f"""
## Metric 4 — Nearest-Neighbor Distance (cm)

| Statistic | Value |
|-----------|-------|
| Mean | {np.mean(nnd_flat):.3f} |
| Median | {np.median(nnd_flat):.3f} |
| Std | {np.std(nnd_flat):.3f} |
| 5th percentile | {np.percentile(nnd_flat, 5):.3f} |
| 25th percentile | {np.percentile(nnd_flat, 25):.3f} |
| 75th percentile | {np.percentile(nnd_flat, 75):.3f} |
| 95th percentile | {np.percentile(nnd_flat, 95):.3f} |

## Figures

- `figures/metric1_polarization.png` — Polarization time series, distribution, density-polarization scatter
- `figures/metric2_speed_turning.png` — Speed and turning-angle distributions by motion state
- `figures/metric3_neighbor_density.png` — Body-centered neighbor density maps
- `figures/metric4_nnd.png` — Nearest-neighbor distance distribution and time series
"""

with open('week1_data_statistics.md', 'w') as f:
    f.write(summary)

print("Saved: week1_data_statistics.md")
print(summary[:500] + "\n...")